# Notebook 25: Adapter Layers - The Original PEFT Method

**Learning Objectives:**
- Understand adapter layers, the pioneering PEFT technique from 2019
- Implement the classic bottleneck adapter architecture
- Compare adapters with LoRA in detail
- Learn when adapters outperform LoRA
- Explore AdapterFusion for multi-task learning

**Prerequisites:** Notebooks 23-24 (LoRA fundamentals)

## 1. Historical Context: Before LoRA, There Were Adapters

### Timeline of PEFT Evolution
```
2019: Adapter Layers (Houlsby et al.) - First practical PEFT
      "Parameter-Efficient Transfer Learning for NLP"
      Problem: Full fine-tuning = millions of parameters per task
      Solution: Insert small bottleneck modules between layers
      Result: 2-5% trainable parameters, 95%+ of full FT performance

2020: Adapter variants proliferate
      - Pfeiffer adapters (more efficient placement)
      - Parallel adapters (speed improvements)

2021: AdapterFusion - multi-task composition
      LoRA emerges as competitor (0.1-1% vs adapters' 2-5%)

2022-2024: LoRA dominates, but adapters remain relevant
      Why? Better modularity, interpretability, task composition
```

### The Original Problem
Pre-2019, transfer learning meant:
- Load pretrained model (e.g., BERT-base: 110M params)
- Fine-tune ALL parameters on your task
- Store separate 110M param model per task
- 10 tasks = 1.1B parameters total (wasteful!)

Adapter solution:
- Store one 110M base model
- Add 2-3M adapter per task
- 10 tasks = 110M + 10×3M = 140M total (92% savings!)

In [ ]:
# Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Optional, List, Dict
import math
from dataclasses import dataclass

# For visualization
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## 2. Classic Adapter Architecture: The Bottleneck Design

### Core Concept
```
Input (d=768)           Original flow, frozen
    |
    v
[Transformer Layer]     <-- Frozen pretrained weights
    |
    +------------------+
    |                  |
    v                  v
  Skip              Adapter Module (trainable)
    |                  |
    |                  v
    |              Down-project: d -> r (r << d)
    |                  |         768 -> 64
    |                  v
    |              Non-linearity (GELU)
    |                  |
    |                  v
    |              Up-project: r -> d
    |                  |       64 -> 768
    |                  |
    +--------+---------+
             |
             v
          Add (residual)
             |
             v
        LayerNorm
```

### Key Design Principles
1. **Bottleneck**: d -> r -> d (r is bottleneck dimension, typically r = d/12)
2. **Residual Connection**: Near-identity initialization (output ≈ 0 at start)
3. **Small Capacity**: Only 2×d×r + 2×r parameters per adapter
4. **Strategic Placement**: After attention and/or FFN blocks

In [ ]:
class AdapterModule(nn.Module):
    """Classic bottleneck adapter (Houlsby et al., 2019)"""
    
    def __init__(
        self,
        d_model: int,           # Model dimension (e.g., 768)
        bottleneck_dim: int,    # Bottleneck size (e.g., 64)
        activation: str = 'gelu',
        init_scale: float = 1e-3  # Near-zero initialization
    ):
        super().__init__()
        self.d_model = d_model
        self.bottleneck_dim = bottleneck_dim
        
        # Down-projection: d -> r
        self.down_project = nn.Linear(d_model, bottleneck_dim)
        
        # Non-linearity
        self.activation = nn.GELU() if activation == 'gelu' else nn.ReLU()
        
        # Up-projection: r -> d
        self.up_project = nn.Linear(bottleneck_dim, d_model)
        
        # Near-zero initialization for residual stability
        # At init, adapter output ≈ 0, so model behaves like frozen base
        nn.init.normal_(self.down_project.weight, std=init_scale)
        nn.init.zeros_(self.down_project.bias)
        nn.init.normal_(self.up_project.weight, std=init_scale)
        nn.init.zeros_(self.up_project.bias)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch, seq_len, d_model)
        Returns:
            adapter_output: (batch, seq_len, d_model)
        """
        # Bottleneck computation
        h = self.down_project(x)      # (B, L, d) -> (B, L, r)
        h = self.activation(h)         # Non-linearity
        h = self.up_project(h)         # (B, L, r) -> (B, L, d)
        return h
    
    def num_parameters(self) -> Dict[str, int]:
        """Count trainable parameters"""
        down_params = self.d_model * self.bottleneck_dim + self.bottleneck_dim
        up_params = self.bottleneck_dim * self.d_model + self.d_model
        return {
            'down_project': down_params,
            'up_project': up_params,
            'total': down_params + up_params
        }


# Test adapter module
d_model = 768
bottleneck = 64
adapter = AdapterModule(d_model, bottleneck)

# Test forward pass
x = torch.randn(2, 128, d_model)  # (batch=2, seq_len=128, d=768)
adapter_out = adapter(x)

print(f"Input shape: {x.shape}")
print(f"Adapter output shape: {adapter_out.shape}")
print(f"\nParameter count: {adapter.num_parameters()}")
print(f"Total: {adapter.num_parameters()['total']:,} parameters")
print(f"\nInitial output magnitude (should be near-zero):")
print(f"  Mean: {adapter_out.abs().mean().item():.6f}")
print(f"  Max:  {adapter_out.abs().max().item():.6f}")

## 3. Where to Insert Adapters: Placement Strategies

### Original Houlsby Adapters (2019)
```
Transformer Layer:
  x = x + Attention(LayerNorm(x))
  x = x + Adapter_1(x)              <-- After attention
  x = x + FFN(LayerNorm(x))
  x = x + Adapter_2(x)              <-- After FFN
```
- 2 adapters per layer
- Maximum expressiveness
- More parameters

### Pfeiffer Adapters (2020)
```
Transformer Layer:
  x = x + Attention(LayerNorm(x))
  x = x + FFN(LayerNorm(x))
  x = x + Adapter(LayerNorm(x))     <-- After FFN only
```
- 1 adapter per layer
- 50% fewer parameters
- Minimal performance drop (<1%)
- Most popular variant

### Parallel Adapters (2021)
```
Transformer Layer:
  attn_out = Attention(LayerNorm(x))
  adapter_out = Adapter(LayerNorm(x))   <-- Parallel to attention
  x = x + attn_out + adapter_out
  # Similar for FFN
```
- Can parallelize computation
- Faster inference
- Slightly different training dynamics

In [ ]:
class TransformerLayerWithAdapters(nn.Module):
    """Transformer layer with configurable adapter placement"""
    
    def __init__(
        self,
        d_model: int = 768,
        n_heads: int = 12,
        d_ff: int = 3072,
        adapter_bottleneck: int = 64,
        adapter_placement: str = 'pfeiffer'  # 'houlsby', 'pfeiffer', 'parallel'
    ):
        super().__init__()
        self.d_model = d_model
        self.adapter_placement = adapter_placement
        
        # Core transformer components (frozen in practice)
        self.ln1 = nn.LayerNorm(d_model)
        self.attention = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )
        
        # Adapter modules (trainable)
        if adapter_placement == 'houlsby':
            self.adapter_after_attn = AdapterModule(d_model, adapter_bottleneck)
            self.adapter_after_ffn = AdapterModule(d_model, adapter_bottleneck)
        elif adapter_placement == 'pfeiffer':
            self.adapter_after_ffn = AdapterModule(d_model, adapter_bottleneck)
        elif adapter_placement == 'parallel':
            self.adapter_parallel_attn = AdapterModule(d_model, adapter_bottleneck)
            self.adapter_parallel_ffn = AdapterModule(d_model, adapter_bottleneck)
        
        # Freeze base model parameters
        for param in [*self.ln1.parameters(), *self.attention.parameters(),
                     *self.ln2.parameters(), *self.ffn.parameters()]:
            param.requires_grad = False
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch, seq_len, d_model)
        """
        if self.adapter_placement == 'houlsby':
            # Attention + adapter
            x = x + self.attention(self.ln1(x), self.ln1(x), self.ln1(x))[0]
            x = x + self.adapter_after_attn(x)
            # FFN + adapter
            x = x + self.ffn(self.ln2(x))
            x = x + self.adapter_after_ffn(x)
            
        elif self.adapter_placement == 'pfeiffer':
            # Attention (no adapter)
            x = x + self.attention(self.ln1(x), self.ln1(x), self.ln1(x))[0]
            # FFN + adapter
            residual = x
            x = self.ln2(x)
            x = residual + self.ffn(x) + self.adapter_after_ffn(x)
            
        elif self.adapter_placement == 'parallel':
            # Parallel attention + adapter
            normed = self.ln1(x)
            x = x + self.attention(normed, normed, normed)[0] + self.adapter_parallel_attn(normed)
            # Parallel FFN + adapter
            normed = self.ln2(x)
            x = x + self.ffn(normed) + self.adapter_parallel_ffn(normed)
        
        return x
    
    def count_trainable_params(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Compare different placements
placements = ['houlsby', 'pfeiffer', 'parallel']
results = []

for placement in placements:
    layer = TransformerLayerWithAdapters(adapter_placement=placement)
    trainable = layer.count_trainable_params()
    total = sum(p.numel() for p in layer.parameters())
    results.append({
        'placement': placement,
        'trainable': trainable,
        'total': total,
        'percent': 100 * trainable / total
    })

print("Adapter Placement Comparison:")
print(f"{'Placement':<15} {'Trainable':>12} {'Total':>12} {'% Trainable':>12}")
print("-" * 55)
for r in results:
    print(f"{r['placement']:<15} {r['trainable']:>12,} {r['total']:>12,} {r['percent']:>11.2f}%")

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
placements_names = [r['placement'].capitalize() for r in results]
trainable_params = [r['trainable'] for r in results]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

bars = ax.bar(placements_names, trainable_params, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('Trainable Parameters', fontsize=12)
ax.set_title('Adapter Placement: Trainable Parameter Count', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bar, val in zip(bars, trainable_params):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:,}',
            ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 4. Adapters vs LoRA: Detailed Comparison

### Architecture Differences

**Adapters:**
```
Original: W @ x
Modified: W @ x + Adapter(x)
          where Adapter(x) = Up(GELU(Down(x)))
```
- Sequential computation (bottleneck)
- Non-linear transformation
- Separate residual path
- Adds latency (extra forward pass)

**LoRA:**
```
Original: W @ x
Modified: (W + BA) @ x = W @ x + BA @ x
```
- Parallel computation (can merge)
- Linear transformation
- Same residual path
- No inference latency (after merge)

### Parameter Efficiency

**Adapters:**
- Parameters: 2 × d × r (+ biases)
- Typical r = d/12 (e.g., 768/12 = 64)
- For d=768, r=64: 2 × 768 × 64 = 98,304 params
- 2-5% of model parameters

**LoRA:**
- Parameters: (d_in + d_out) × r
- Typical r = 8-16 (much lower!)
- For d=768, r=8: (768 + 768) × 8 = 12,288 params
- 0.1-1% of model parameters
- 8x fewer parameters than adapters!

In [ ]:
# Quantitative comparison
def compare_peft_methods(d_model=768, n_layers=12):
    """
    Compare adapters vs LoRA for a BERT-like model
    """
    # Base model parameters (approx)
    # Attention: 4 × d × d (Q, K, V, O projections)
    # FFN: 2 × d × 4d
    base_params_per_layer = 4 * d_model**2 + 2 * d_model * (4 * d_model)
    base_total = n_layers * base_params_per_layer
    
    results = []
    
    # Houlsby adapters (2 per layer)
    adapter_r = d_model // 12
    adapter_params_per_module = 2 * d_model * adapter_r
    houlsby_total = n_layers * 2 * adapter_params_per_module
    results.append({
        'method': 'Houlsby Adapters',
        'params': houlsby_total,
        'percent': 100 * houlsby_total / base_total,
        'bottleneck_dim': adapter_r
    })
    
    # Pfeiffer adapters (1 per layer)
    pfeiffer_total = n_layers * adapter_params_per_module
    results.append({
        'method': 'Pfeiffer Adapters',
        'params': pfeiffer_total,
        'percent': 100 * pfeiffer_total / base_total,
        'bottleneck_dim': adapter_r
    })
    
    # LoRA (on Q, V projections only - common config)
    lora_r = 8
    lora_params_per_matrix = (d_model + d_model) * lora_r  # A + B
    lora_total = n_layers * 2 * lora_params_per_matrix  # Q and V in each layer
    results.append({
        'method': 'LoRA (r=8)',
        'params': lora_total,
        'percent': 100 * lora_total / base_total,
        'bottleneck_dim': lora_r
    })
    
    # LoRA (higher rank)
    lora_r = 64
    lora_params_per_matrix = (d_model + d_model) * lora_r
    lora_64_total = n_layers * 2 * lora_params_per_matrix
    results.append({
        'method': 'LoRA (r=64)',
        'params': lora_64_total,
        'percent': 100 * lora_64_total / base_total,
        'bottleneck_dim': lora_r
    })
    
    return base_total, results


base_params, comparison = compare_peft_methods()

print(f"Base Model: ~{base_params:,} parameters\n")
print("PEFT Method Comparison:")
print(f"{'Method':<25} {'Trainable Params':>20} {'% of Base':>12} {'Bottleneck':>12}")
print("-" * 72)
for r in comparison:
    print(f"{r['method']:<25} {r['params']:>20,} {r['percent']:>11.2f}% {r['bottleneck_dim']:>12}")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Absolute parameters
methods = [r['method'] for r in comparison]
params = [r['params'] for r in comparison]
colors = ['#FF6B6B', '#4ECDC4', '#95E1D3', '#F38181']

bars1 = ax1.bar(range(len(methods)), params, color=colors, alpha=0.7, edgecolor='black')
ax1.set_xticks(range(len(methods)))
ax1.set_xticklabels(methods, rotation=15, ha='right')
ax1.set_ylabel('Trainable Parameters', fontsize=12)
ax1.set_title('PEFT Methods: Absolute Parameter Count', fontsize=13, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

for bar, val in zip(bars1, params):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{val/1e6:.2f}M',
            ha='center', va='bottom', fontsize=9)

# Percentage of base model
percents = [r['percent'] for r in comparison]
bars2 = ax2.bar(range(len(methods)), percents, color=colors, alpha=0.7, edgecolor='black')
ax2.set_xticks(range(len(methods)))
ax2.set_xticklabels(methods, rotation=15, ha='right')
ax2.set_ylabel('% of Base Model', fontsize=12)
ax2.set_title('PEFT Methods: Efficiency Comparison', fontsize=13, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

for bar, val in zip(bars2, percents):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.2f}%',
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("\nKey Insight: LoRA (r=8) uses 5-10x fewer parameters than adapters!")

## 5. When Adapters Outperform LoRA

### Scenario 1: Interpretability & Modularity
**Adapters win because:**
- Each adapter = isolated module for specific task
- Easy to inspect, swap, or remove
- Can analyze adapter activations separately
- LoRA: merged into weight matrix, hard to disentangle

**Example use case:**
```
Research: "Which layer adapters contribute most to task X?"
Answer: Remove adapters layer-by-layer, measure performance drop
With LoRA: Would need to retrain from scratch
```

### Scenario 2: Multi-Task Composition (AdapterFusion)
**Adapters win because:**
- Can compose multiple adapters at inference
- Learn weighted combination of task-specific knowledge
- LoRA: merging multiple LoRAs is less flexible

### Scenario 3: Non-Linear Task Adaptations
**Adapters win because:**
- Bottleneck + non-linearity = more expressive
- LoRA = purely linear transformation
- For tasks requiring non-linear adaptation, adapters > LoRA

### Scenario 4: Very Large Models with Small r
**Adapters can win because:**
- LoRA with tiny r (4-8) might be too constrained
- Adapter bottleneck can be larger (64) while staying efficient
- Non-linearity compensates for lower rank

### When LoRA Wins (for context)
- Inference speed critical (LoRA merges, zero overhead)
- Extreme parameter efficiency needed (LoRA uses 5-10x fewer)
- Simple deployment (single model file, no adapter management)
- Most production scenarios

In [ ]:
# Demonstrate non-linear expressiveness
def measure_expressiveness(method='adapter', rank=8, d_model=512, num_samples=1000):
    """
    Measure how well each method can approximate a non-linear function
    """
    # Generate synthetic non-linear target
    torch.manual_seed(42)
    X = torch.randn(num_samples, d_model)
    # Non-linear target: tanh(X @ W1) @ W2
    W1 = torch.randn(d_model, 64) * 0.1
    W2 = torch.randn(64, d_model) * 0.1
    Y_target = torch.tanh(X @ W1) @ W2
    
    if method == 'adapter':
        model = AdapterModule(d_model, rank)
    else:  # LoRA approximation
        class LoRAModule(nn.Module):
            def __init__(self, d, r):
                super().__init__()
                self.A = nn.Parameter(torch.randn(d, r) * 0.01)
                self.B = nn.Parameter(torch.randn(r, d) * 0.01)
            
            def forward(self, x):
                return x @ self.A @ self.B
        
        model = LoRAModule(d_model, rank)
    
    # Train briefly
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    losses = []
    
    for epoch in range(100):
        optimizer.zero_grad()
        Y_pred = model(X)
        loss = F.mse_loss(Y_pred, Y_target)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    
    return losses


# Compare expressiveness
print("Testing non-linear expressiveness...\n")
adapter_losses = measure_expressiveness('adapter', rank=64)
lora_losses = measure_expressiveness('lora', rank=64)

print(f"Final Loss (Adapter r=64): {adapter_losses[-1]:.6f}")
print(f"Final Loss (LoRA r=64):    {lora_losses[-1]:.6f}")
print(f"\nAdapter advantage: {(lora_losses[-1] / adapter_losses[-1] - 1) * 100:.1f}% lower loss")

# Visualize
plt.figure(figsize=(10, 6))
plt.plot(adapter_losses, label='Adapter (r=64)', linewidth=2, color='#FF6B6B')
plt.plot(lora_losses, label='LoRA (r=64)', linewidth=2, color='#4ECDC4')
plt.xlabel('Training Steps', fontsize=12)
plt.ylabel('MSE Loss (log scale)', fontsize=12)
plt.yscale('log')
plt.title('Non-Linear Function Approximation: Adapter vs LoRA', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nInterpretation: Adapters converge faster on non-linear targets")
print("due to GELU non-linearity. LoRA is limited to linear transformations.")

## 6. AdapterFusion: Sequential Adapters for Multi-Task Learning

### The Multi-Task Challenge
```
Problem: Train model on N tasks
  - Task A: Sentiment analysis
  - Task B: NER
  - Task C: Question answering

Naive approach:
  - Train 3 separate adapters
  - At inference, use adapter for specific task
  - No knowledge sharing between tasks!

AdapterFusion approach (Pfeiffer et al., 2020):
  1. Train task-specific adapters (frozen after training)
  2. Learn FUSION layer that combines adapter outputs
  3. Fusion learns which adapter to attend to
  4. Result: multi-task model with task composition
```

### Architecture
```
                Input x
                  |
       +----------+----------+
       |          |          |
       v          v          v
  Adapter_A  Adapter_B  Adapter_C  (frozen)
       |          |          |
       +----------+----------+
                  |
                  v
          Fusion Layer (trainable)
            Attention over adapters
                  |
                  v
        Weighted combination
                  |
                  v
              Output
```

Fusion attention:
```
Query = Linear(x)
Keys = [Linear(adapter_i(x)) for all adapters]
Attention = softmax(Query @ Keys^T)
Output = sum(Attention[i] * adapter_i(x))
```

In [ ]:
class AdapterFusion(nn.Module):
    """Combine multiple task-specific adapters with learned attention"""
    
    def __init__(
        self,
        adapters: List[AdapterModule],
        d_model: int,
        fusion_type: str = 'attention'  # 'attention' or 'weighted_sum'
    ):
        super().__init__()
        self.adapters = nn.ModuleList(adapters)
        self.d_model = d_model
        self.n_adapters = len(adapters)
        self.fusion_type = fusion_type
        
        # Freeze adapter parameters
        for adapter in self.adapters:
            for param in adapter.parameters():
                param.requires_grad = False
        
        if fusion_type == 'attention':
            # Query projection from input
            self.query_proj = nn.Linear(d_model, d_model)
            # Key projections for each adapter output
            self.key_proj = nn.Linear(d_model, d_model)
            # Value projections
            self.value_proj = nn.Linear(d_model, d_model)
            self.scale = 1.0 / math.sqrt(d_model)
        else:  # weighted_sum
            # Learnable weights for each adapter
            self.adapter_weights = nn.Parameter(torch.ones(self.n_adapters))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch, seq_len, d_model)
        Returns:
            fused_output: (batch, seq_len, d_model)
        """
        # Get outputs from all adapters
        adapter_outputs = [adapter(x) for adapter in self.adapters]  # List of (B, L, d)
        adapter_outputs = torch.stack(adapter_outputs, dim=2)  # (B, L, n_adapters, d)
        
        if self.fusion_type == 'attention':
            # Compute attention over adapters
            query = self.query_proj(x)  # (B, L, d)
            keys = self.key_proj(adapter_outputs)  # (B, L, n_adapters, d)
            values = self.value_proj(adapter_outputs)  # (B, L, n_adapters, d)
            
            # Attention scores
            scores = torch.einsum('bld,blnd->bln', query, keys) * self.scale  # (B, L, n_adapters)
            attention = F.softmax(scores, dim=-1)  # (B, L, n_adapters)
            
            # Weighted combination
            output = torch.einsum('bln,blnd->bld', attention, values)  # (B, L, d)
            
        else:  # weighted_sum
            # Simple weighted average
            weights = F.softmax(self.adapter_weights, dim=0)  # (n_adapters,)
            output = torch.einsum('n,blnd->bld', weights, adapter_outputs)  # (B, L, d)
        
        return output
    
    def get_adapter_weights(self, x: torch.Tensor) -> torch.Tensor:
        """Return attention weights over adapters for analysis"""
        with torch.no_grad():
            adapter_outputs = [adapter(x) for adapter in self.adapters]
            adapter_outputs = torch.stack(adapter_outputs, dim=2)
            
            if self.fusion_type == 'attention':
                query = self.query_proj(x)
                keys = self.key_proj(adapter_outputs)
                scores = torch.einsum('bld,blnd->bln', query, keys) * self.scale
                attention = F.softmax(scores, dim=-1)
                return attention.mean(dim=(0, 1))  # Average over batch and sequence
            else:
                return F.softmax(self.adapter_weights, dim=0)


# Demo: Create multi-task fusion
d_model = 512
bottleneck = 64

# Create 3 "task-specific" adapters (pretend they're pre-trained)
adapter_sentiment = AdapterModule(d_model, bottleneck)
adapter_ner = AdapterModule(d_model, bottleneck)
adapter_qa = AdapterModule(d_model, bottleneck)

adapters = [adapter_sentiment, adapter_ner, adapter_qa]
task_names = ['Sentiment', 'NER', 'QA']

# Create fusion layer
fusion = AdapterFusion(adapters, d_model, fusion_type='attention')

# Test forward pass
x = torch.randn(4, 64, d_model)
fused_output = fusion(x)

print(f"Input shape: {x.shape}")
print(f"Fused output shape: {fused_output.shape}")
print(f"\nTrainable parameters in fusion: {sum(p.numel() for p in fusion.parameters() if p.requires_grad):,}")
print(f"Frozen parameters (adapters): {sum(p.numel() for p in fusion.parameters() if not p.requires_grad):,}")

# Analyze adapter weights
weights = fusion.get_adapter_weights(x)
print(f"\nAdapter attention weights:")
for name, weight in zip(task_names, weights):
    print(f"  {name}: {weight.item():.3f}")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Adapter weights
colors = ['#FF6B6B', '#4ECDC4', '#95E1D3']
bars = ax1.bar(task_names, weights.numpy(), color=colors, alpha=0.7, edgecolor='black')
ax1.set_ylabel('Attention Weight', fontsize=12)
ax1.set_title('AdapterFusion: Learned Task Weights', fontsize=13, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim([0, 1])

for bar, val in zip(bars, weights):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.3f}',
            ha='center', va='bottom', fontsize=10)

# Architecture diagram (text-based)
ax2.axis('off')
architecture_text = """
AdapterFusion Architecture:

          Input (B, L, d)
                |
    +-----------+-----------+
    |           |           |
    v           v           v
Adapter_1   Adapter_2   Adapter_3
 (frozen)    (frozen)    (frozen)
    |           |           |
    +-----------+-----------+
                |
                v
      Attention Fusion Layer
         (trainable)
                |
                v
      Weighted Combination
                |
                v
         Output (B, L, d)

Benefits:
• Compose multiple task adapters
• Learn task importance dynamically
• Transfer across task boundaries
• Interpretable attention weights
"""
ax2.text(0.1, 0.5, architecture_text, fontsize=10, family='monospace',
         verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.tight_layout()
plt.show()

## 7. Practical Implementation with Hugging Face PEFT

The PEFT library supports adapters out of the box:

In [ ]:
# Note: This is example code structure - requires PEFT library installation
"""
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import get_peft_model, AdapterConfig, TaskType

# Load base model
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# Configure adapters
adapter_config = AdapterConfig(
    peft_type="BOTTLENECK_ADAPTER",  # Classic adapter
    task_type=TaskType.SEQ_CLS,
    reduction_factor=16,  # d_model / bottleneck_dim
    non_linearity="gelu",
    adapter_dropout=0.1,
    original_ln_after=True,
    ln_after=True,
    ln_before=False,
    mh_adapter=True,  # Add adapter after multi-head attention
    output_adapter=True,  # Add adapter after FFN
    # mh_adapter=True, output_adapter=True => Houlsby adapters
    # mh_adapter=False, output_adapter=True => Pfeiffer adapters
)

# Wrap model with adapters
model = get_peft_model(model, adapter_config)

# Print trainable parameters
model.print_trainable_parameters()
# Output: trainable params: 894,528 || all params: 109,483,778 || trainable%: 0.82

# Train as normal
# ...

# Save only adapter weights
model.save_pretrained("./sentiment_adapter")
# Saves only ~3MB instead of 440MB!

# Load adapter later
from peft import PeftModel
base_model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased")
model = PeftModel.from_pretrained(base_model, "./sentiment_adapter")
"""

print("Adapter implementation with PEFT library (see code comments)")
print("\nKey configuration options:")
print("  - reduction_factor: Controls bottleneck size (higher = smaller adapter)")
print("  - mh_adapter: Add adapter after attention (Houlsby-style)")
print("  - output_adapter: Add adapter after FFN")
print("  - Both True => Houlsby adapters (2 per layer)")
print("  - Only output_adapter => Pfeiffer adapters (1 per layer)")

## 8. Comprehensive Comparison: Adapters vs LoRA Trade-offs

### Parameter Efficiency
- Winner: LoRA (5-10x fewer parameters)
- LoRA: 0.1-1% of base model
- Adapters: 2-5% of base model

### Inference Speed
- Winner: LoRA (can merge, zero overhead)
- LoRA merged: Same speed as base model
- Adapters: +10-20% latency (extra forward pass)

### Training Speed
- Winner: Tie
- Both similar (adapters slightly slower due to non-linearity)

### Expressiveness
- Winner: Adapters (non-linear transformations)
- Adapters: Bottleneck + GELU = more expressive
- LoRA: Purely linear (limited for non-linear adaptations)

### Modularity
- Winner: Adapters (isolated components)
- Adapters: Easy to swap, remove, inspect
- LoRA: Merged into weights, harder to separate

### Multi-Task Composition
- Winner: Adapters (AdapterFusion)
- Adapters: Can learn attention over multiple task adapters
- LoRA: Merging multiple LoRAs less flexible

### Deployment Simplicity
- Winner: LoRA (single merged model)
- LoRA: Merge and deploy as standard model
- Adapters: Need to manage base + adapter modules

### Memory (Training)
- Winner: LoRA (fewer gradients)
- Fewer parameters = less gradient memory

### Memory (Inference)
- Winner: LoRA (after merge)
- LoRA merged: Same as base model
- Adapters: Base + adapter weights in memory

### Research & Analysis
- Winner: Adapters (interpretability)
- Easier to study individual adapter contributions
- Better for ablation studies

### Production Use
- Winner: LoRA (efficiency + simplicity)
- Most production systems choose LoRA
- Better performance/cost ratio

In [ ]:
# Comprehensive comparison visualization
comparison_metrics = {
    'Metric': [
        'Parameter\nEfficiency',
        'Inference\nSpeed',
        'Training\nSpeed',
        'Expressiveness',
        'Modularity',
        'Multi-Task\nComposition',
        'Deployment\nSimplicity',
        'Memory\n(Training)',
        'Memory\n(Inference)',
        'Interpretability'
    ],
    'LoRA': [10, 10, 5, 4, 4, 4, 10, 9, 10, 4],
    'Adapters': [5, 4, 5, 8, 9, 9, 5, 5, 5, 9]
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Radar chart
categories = comparison_metrics['Metric']
lora_scores = comparison_metrics['LoRA']
adapter_scores = comparison_metrics['Adapters']

angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
lora_scores_plot = lora_scores + lora_scores[:1]
adapter_scores_plot = adapter_scores + adapter_scores[:1]
angles_plot = angles + angles[:1]

ax1 = plt.subplot(121, projection='polar')
ax1.plot(angles_plot, lora_scores_plot, 'o-', linewidth=2, label='LoRA', color='#4ECDC4')
ax1.fill(angles_plot, lora_scores_plot, alpha=0.25, color='#4ECDC4')
ax1.plot(angles_plot, adapter_scores_plot, 'o-', linewidth=2, label='Adapters', color='#FF6B6B')
ax1.fill(angles_plot, adapter_scores_plot, alpha=0.25, color='#FF6B6B')
ax1.set_xticks(angles)
ax1.set_xticklabels(categories, size=9)
ax1.set_ylim(0, 10)
ax1.set_yticks([2, 4, 6, 8, 10])
ax1.set_yticklabels(['2', '4', '6', '8', '10'], size=8)
ax1.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)
ax1.set_title('LoRA vs Adapters: Multi-Dimensional Comparison', 
              fontsize=13, fontweight='bold', pad=20)
ax1.grid(True)

# Bar chart comparison
ax2 = plt.subplot(122)
x = np.arange(len(categories))
width = 0.35

bars1 = ax2.barh(x - width/2, lora_scores, width, label='LoRA', 
                 color='#4ECDC4', alpha=0.8, edgecolor='black')
bars2 = ax2.barh(x + width/2, adapter_scores, width, label='Adapters',
                 color='#FF6B6B', alpha=0.8, edgecolor='black')

ax2.set_yticks(x)
ax2.set_yticklabels(categories, fontsize=9)
ax2.set_xlabel('Score (0-10)', fontsize=11)
ax2.set_title('Head-to-Head Comparison', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(axis='x', alpha=0.3)
ax2.set_xlim(0, 11)

plt.tight_layout()
plt.show()

# Calculate overall scores
lora_total = sum(lora_scores)
adapter_total = sum(adapter_scores)

print("\nOverall Scores:")
print(f"  LoRA:     {lora_total}/100 ({lora_total/len(lora_scores):.1f}/10 average)")
print(f"  Adapters: {adapter_total}/100 ({adapter_total/len(adapter_scores):.1f}/10 average)")
print("\nConclusion: Choose based on priorities:")
print("  - Production/efficiency priority => LoRA")
print("  - Research/interpretability priority => Adapters")
print("  - Multi-task composition => Adapters")
print("  - Extreme parameter efficiency => LoRA")

## Summary: Adapter Layers in PEFT

### Key Takeaways

1. **Historical Significance**
   - First practical PEFT method (2019)
   - Enabled efficient multi-task learning
   - Paved the way for LoRA and other methods

2. **Core Architecture**
   - Bottleneck design: d → r → d (r << d)
   - Non-linear transformation (GELU)
   - Near-zero initialization for stability
   - 2-5% trainable parameters

3. **Placement Strategies**
   - Houlsby: 2 adapters/layer (after attn + FFN)
   - Pfeiffer: 1 adapter/layer (after FFN only) ← most popular
   - Parallel: Compute adapters in parallel with attention

4. **Adapters vs LoRA**
   - LoRA: 5-10x fewer parameters, faster inference
   - Adapters: More expressive (non-linear), better modularity
   - LoRA wins: production, efficiency
   - Adapters win: research, interpretability, multi-task

5. **AdapterFusion**
   - Compose multiple task adapters
   - Learn attention over adapter outputs
   - Enable knowledge transfer across tasks
   - Superior to single-task adapters

6. **When to Use Adapters**
   - Need interpretability/ablation studies
   - Multi-task composition required
   - Non-linear adaptations important
   - Research over production focus

7. **When to Use LoRA Instead**
   - Production deployment
   - Inference speed critical
   - Extreme parameter efficiency needed
   - Simple deployment workflow

### Practical Recommendations

```
Decision Tree:

Is this for production?
├─ Yes → LoRA (efficiency + speed)
└─ No → Continue

Need multi-task composition?
├─ Yes → Adapters + AdapterFusion
└─ No → Continue

Need interpretability/ablations?
├─ Yes → Adapters (modular)
└─ No → LoRA (simpler)

Non-linear adaptations critical?
├─ Yes → Adapters (GELU)
└─ No → LoRA (sufficient)
```

### Next Steps
- Notebook 26: Prompt tuning (even more efficient: <0.01%!)
- Notebook 27: Comprehensive PEFT benchmark
- Notebook 28: LoRA merging strategies
- Notebook 29: Advanced LoRA variants (DoRA, AdaLoRA, etc.)

### Further Reading
- Houlsby et al. (2019): "Parameter-Efficient Transfer Learning for NLP"
- Pfeiffer et al. (2020): "AdapterFusion: Non-Destructive Task Composition"
- He et al. (2021): "Towards a Unified View of Parameter-Efficient Transfer Learning"
- Comparison: https://arxiv.org/abs/2110.04366